<a href="https://colab.research.google.com/github/RashiBista/GraphRAG/blob/main/03_extract_2wiki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install groq

In [2]:
import json, re, time
from pathlib import Path
from google.colab import drive, userdata
from groq import Groq, RateLimitError

In [3]:
drive.mount("/content/drive")
WORKDIR = Path("/content/drive/MyDrive/HybridGraphRAG/twowiki")

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

chunks = load_jsonl(WORKDIR / "document_chunks.jsonl")
questions = load_jsonl(WORKDIR / "questions.jsonl")

groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
LLM = "openai/gpt-oss-20b"   # the real extraction model, not the safeguard one

def chat_with_retry(**kwargs):
    for attempt in range(8):
        try:
            return groq_client.chat.completions.create(**kwargs)
        except RateLimitError:
            wait = 3 * (attempt + 1)
            print(f"  rate limited, waiting {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after 8 retries")

print(len(chunks), "chunks,", len(questions), "questions")

Mounted at /content/drive
182 chunks, 20 questions


In [4]:
ENTITY_TYPES = {
    "PERSON":       "a real person",
    "WORK":         "a film, song, book, album or other creative work",
    "PLACE":        "a city, region, country or other location",
    "ORGANIZATION": "a company, school, band, club or institution",
    "AWARD":        "a prize or honour",
    "DATE":         "a date or year, copied as written in the text",
    "OTHER":        "important, but fits none of the above",
}

# Direction rule: (source, RELATION, target). The definitions say who is who.
RELATION_TYPES = {
    # links between things (these are what connect articles to each other)
    "DIRECTOR":               "target directed source (source is a film)",
    "FATHER":                 "target is the father of source",
    "MOTHER":                 "target is the mother of source",
    "SPOUSE":                 "target is married to source",
    "CHILD":                  "target is a child of source",
    "SIBLING":                "target is a brother or sister of source",
    "PERFORMER":              "target performed source (e.g. the singer of a song)",
    "COMPOSER":               "target composed source",
    "EDUCATED_AT":            "source studied at target",
    "EMPLOYER":               "source works or worked for target",
    "AWARD_RECEIVED":         "source received target (an award)",
    "PLACE_OF_BIRTH":         "source was born in target",
    "PLACE_OF_DEATH":         "source died in target",
    "PLACE_OF_BURIAL":        "source was buried in target",
    "COUNTRY_OF_CITIZENSHIP": "source is a citizen of target",
    "COUNTRY_OF_ORIGIN":      "source (a work) comes from target (a country)",
    "COUNTRY":                "source is located in target (a country)",
    # facts whose target is a DATE
    "DATE_OF_BIRTH":          "source was born on target (a DATE)",
    "DATE_OF_DEATH":          "source died on target (a DATE)",
    "PUBLICATION_DATE":       "source was released or published on target (a DATE)",
    "INCEPTION":              "source was founded or created on target (a DATE)",
    # catch-all
    "RELATED_TO":             "a clear link that fits none of the above",
}
print(len(ENTITY_TYPES), "entity types,", len(RELATION_TYPES), "relation types")

7 entity types, 22 relation types


In [5]:
import json, re

def build_schema(entity_types, relation_types):
    return {
        "type": "object",
        "properties": {
            "entities": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "type": {"type": "string", "enum": list(entity_types)},
                    },
                    "required": ["name", "type"],
                    "additionalProperties": False,
                },
            },
            "relationships": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "source": {"type": "string"},
                        "relation": {"type": "string", "enum": list(relation_types)},
                        "target": {"type": "string"},
                    },
                    "required": ["source", "relation", "target"],
                    "additionalProperties": False,
                },
            },
        },
        "required": ["entities", "relationships"],
        "additionalProperties": False,
    }

SCHEMA = build_schema(ENTITY_TYPES, RELATION_TYPES)

def build_prompt(text):
    types_txt = "\n".join(f"- {k}: {v}" for k, v in ENTITY_TYPES.items())
    rels_txt = "\n".join(f"- {k}: {v}" for k, v in RELATION_TYPES.items())
    return f"""You are an information extraction system. Build a knowledge graph from the encyclopedia-style TEXT below.

The TEXT begins with the title of its article ("Title: ..."). The article is about the entity named in the title, so "he", "she", "it", "the film" or "the band" refer to that entity.

ENTITY TYPES:
{types_txt}

RELATIONS, written as (source, RELATION, target). Use only these:
{rels_txt}

Rules:
1. Extract only what the TEXT explicitly states. Never use outside knowledge, even if you know more about the subject.
2. Entity names are noun phrases taken from the text or the title, using the fullest name given. Never use verbs or actions.
3. Follow the relation definitions exactly. They say which way each relation points.
4. Dates are DATE entities, copied as written.
5. Every source and target of a relationship must also appear in the entities list.
6. Prefer a specific relation over RELATED_TO. If nothing fits, return empty lists.

TEXT:
{text}
"""

def extract(text, model=None):
    response = chat_with_retry(
        model=model or LLM,
        messages=[{"role": "user", "content": build_prompt(text)}],
        temperature=0,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "graph_extraction", "strict": True, "schema": SCHEMA},
        },
    )
    return json.loads(response.choices[0].message.content)

def validate_extraction(data):
    if not isinstance(data, dict):
        return False
    if not isinstance(data.get("entities"), list) or not isinstance(data.get("relationships"), list):
        return False
    for e in data["entities"]:
        if not isinstance(e, dict) or "name" not in e or "type" not in e:
            return False
    for r in data["relationships"]:
        if not isinstance(r, dict) or not {"source", "relation", "target"} <= r.keys():
            return False
    return True

def normalize_entity_name(name):
    return re.sub(r"\s+", " ", str(name)).strip()

def normalize_extraction(data):
    entities, relationships = [], []
    seen_e, seen_r = set(), set()
    for e in data.get("entities", []):
        name = normalize_entity_name(e["name"])
        etype = str(e["type"]).strip().upper()
        if name and (name.lower(), etype) not in seen_e:
            entities.append({"name": name, "type": etype})
            seen_e.add((name.lower(), etype))
    for r in data.get("relationships", []):
        s = normalize_entity_name(r["source"])
        rel = str(r["relation"]).strip().upper()
        t = normalize_entity_name(r["target"])
        if s and t and rel and (s.lower(), rel, t.lower()) not in seen_r:
            relationships.append({"source": s, "relation": rel, "target": t})
            seen_r.add((s.lower(), rel, t.lower()))
    return {"entities": entities, "relationships": relationships}

In [6]:
TEST_TITLES = ["Naïma Ben Ali", "Zine El Abidine Ben Ali", "Gyrid of Sweden", "Harald Bluetooth"]
by_title = {c["title"]: c for c in chunks}

def show_extraction(ch):
    out = normalize_extraction(extract(ch["text"]))
    print(f"\n===== {ch['title']}  [{ch['id']}] =====")
    print("Valid shape:", validate_extraction(out))
    print("ENTITIES:", "; ".join(f"{e['name']} [{e['type']}]" for e in out["entities"]))
    for r in out["relationships"]:
        print(f"  {r['source']} --{r['relation']}--> {r['target']}")

for t in TEST_TITLES:
    show_extraction(by_title[t])

print("\n\n##### GOLD TRIPLES from the questions that use these paragraphs #####")
seen = set()
for q in questions:
    if any(t in q["gold_titles"] for t in TEST_TITLES) and q["id"] not in seen:
        seen.add(q["id"])
        print(f"\n{q['question']}")
        for e in q["evidences"]:
            print("   ", e)


===== Naïma Ben Ali  [0101_na_ma_ben_ali] =====
Valid shape: True
ENTITIES: Naïma Ben Ali [PERSON]; Zine El Abidine Ben Ali [PERSON]
  Naïma Ben Ali --SPOUSE--> Zine El Abidine Ben Ali


BadRequestError: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}